# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR^2 protein abundance dataset using the `mlcroissant` library, referencing all key dataset components by their `@id`.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
```
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json
```

In [ ]:
# Ensure `mlcroissant` is installed in this environment
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and records with croissant.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")
print(f"Version: {metadata.version}")
print(f"Published: {metadata.datePublished}")


## 2. Data Overview
Review all available record sets and fields, referencing each by its `@id`. This helps identify how the data is subdivided. Here we list each `@id` for `RecordSet` and their associated field `@id`s.

In [ ]:
# List all RecordSet and Field IDs in this dataset
recordsets = list(metadata.record_sets)
if len(recordsets) == 0:
    print("No explicit record sets found in root metadata; trying to find some in distribution...")
    # Sometimes the Croissant schema only links data via distribution/fileObject
    # You may need to manually locate record set IDs.
    recordsets = []
    if hasattr(metadata, 'distribution') and metadata.distribution:
        for dist in metadata.distribution:
            if hasattr(dist, 'record_sets'):
                recordsets.extend(dist.record_sets)
else:
    print(f"Found {len(recordsets)} record set(s):")
    for rs in recordsets:
        print(f" - RecordSet @id: {rs.id} | name: {rs.name if hasattr(rs, 'name') else 'N/A'}")
        # List all fields/columns for this RecordSet
        print("   Fields/Columns IDs:")
        for field in rs.fields:
            print(f"      - {field.id} ({field.name})")

if not recordsets:
    # If still empty, try top-level fallback for one record set
    print("No record sets in schema (possibly single file, single table). Attempting to infer dataset records...")
    # Some datasets are flat; mlcroissant will default to one record set.


## 3. Data Extraction
Load the main data table into a DataFrame for analysis. Use record set and field `@id` values from the overview above.

For this dataset, we identify the `RecordSet` (or records) by its `@id` for reproducibility.

In [ ]:
# If record set list is empty, let’s use the mlcroissant interface to see all top-level record sets.
recordset_ids = [rs.id for rs in getattr(metadata, 'record_sets', [])]
if not recordset_ids:
    # If not structured with explicit record sets, use default: the Croissant DataFrame should load everything in one
    print("No record sets; using default implicit record set 'records'.")
    recordset_ids = ['records']

# Load all record sets
dataframes = {}
for recordset_id in recordset_ids:
    print(f"Loading records for RecordSet @id: {recordset_id}")
    records = list(dataset.records(record_set=recordset_id))
    dataframes[recordset_id] = pd.DataFrame(records)

# Display columns for first record set
selected_recordset = recordset_ids[0]
print(f"Columns for RecordSet '{selected_recordset}':")
print(dataframes[selected_recordset].columns.tolist())
dataframes[selected_recordset].head()

## 4. Exploratory Data Analysis (EDA)
Let's process and inspect the data. All references to fields/columns will use their `@id`, as recommended by the Croissant specification.

We'll:
- Filter records by a numeric field (e.g., `coverage_percent` or `molecular_weight`),
- Normalize the selected field,
- Group by a categorical field (e.g., by description or protein accession if available),
- Show a preview of results.


In [ ]:
# Adapt the field IDs based on those discovered above. (Example field IDs used for illustration; update to actual dataset field @ids)
df = dataframes[selected_recordset]

# Print a sample of available columns and infer likely numeric and categorical fields
print("Top-level DataFrame columns:")
print(df.columns.tolist())

# Try to auto-detect possible numeric and grouping fields by inspecting DataFrame
numeric_candidates = df.select_dtypes(include=[np.number]).columns.tolist()
print("Numeric candidates (auto-detected):", numeric_candidates)

# You may need to adjust these IDs to match real field @id (for example, 'coverage_percent' or similar from data overview)
numeric_field = numeric_candidates[0] if numeric_candidates else None
group_candidates = [col for col in df.columns if df[col].dtype == object and not df[col].astype(str).str.isnumeric().all()]
group_field = group_candidates[0] if group_candidates else None

if numeric_field is None:
    print("No numeric field detected for EDA.")
else:
    threshold = df[numeric_field].mean()  # Use mean as a default threshold
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    print(filtered_df.head())

    filtered_df = filtered_df.copy()
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    if group_field is not None:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped data by {group_field}:")
        print(grouped_df.head())

## 5. Visualization
Visualize the distribution of the selected numeric field and relationship with the group field (if present).


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if 'filtered_df' in locals() and numeric_field is not None:
    plt.figure(figsize=(8, 5))
    sns.histplot(filtered_df[numeric_field], bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    if group_field is not None and group_field in filtered_df:
        plt.figure(figsize=(10, 6))
        order = filtered_df[group_field].value_counts().index[:10]
        sns.boxplot(
            x=group_field,
            y=numeric_field,
            data=filtered_df[filtered_df[group_field].isin(order)],
        )
        plt.xticks(rotation=45)
        plt.title(f"{numeric_field} distribution by {group_field} (top 10)")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()


## 6. Conclusion
In this exploration, we loaded and processed the FAIR^2 mass spectrometry protein dataset using the `mlcroissant` library, following all Croissant entity conventions by referencing core objects by their `@id`. Our workflow demonstrated programmatic steps to load metadata and records, dynamically browse field structure, filter and normalize quantities, and visualize results, all with reproducibility.

**Please refer to documentation or the printed field lists above to select further record sets or fields for deeper exploration of specific attributes.**